# Financial Analysis

In [82]:
# imports
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt
import numpy as np

In [83]:
# loading data
gates = pd.read_csv("../data/processed/gates_clean.csv")

In [84]:
flight_costs = pd.read_csv("../data/processed/flight_costs_df.csv")

In [85]:
bookings = pd.read_csv("../data/processed/bookings.csv")

In [86]:
bookings_routes = pd.read_csv("../data/processed/booking_routes.csv")

In [87]:
gates.head(5)

,City,Gate_Number,Route_ID,Year,Annual_Gate_Cost
0,Abbotsford,G1,YXX-YYC,2024-2025,166800
1,Calgary,B1,YYC-YVR,2024-2025,373200
2,Calgary,B1,YYC-YYZ,2024-2025,373200
3,Calgary,B1,YYC-YEG,2024-2025,373200
4,Calgary,B2,YYC-YWG,2024-2025,351900


In [88]:
flight_costs.head(5)

,Cost_Date,Route_ID,From,To,Crew,Food,Fuel,Maintenance,Total
0,2024-09-01 06:30:00,YVR-YLW,Vancouver,Kelowna,1388.53,170.52,1401.75,1657.45,4618.25
1,2024-09-01 06:35:00,YLW-YYC,Kelowna,Calgary,1673.28,221.60,1800.44,1898.61,5593.93
2,2024-09-01 06:45:00,YYZ-YYC,Toronto,Calgary,12105.73,2037.11,16016.15,9391.66,39550.65
3,2024-09-01 06:50:00,YLW-YVR,Kelowna,Vancouver,1458.14,187.03,1532.94,1540.96,4719.07
4,2024-09-01 06:50:00,YQR-YYC,Regina,Calgary,2745.73,333.72,2923.61,2504.61,8507.67


In [89]:
bookings.head(9)

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.00,0.0,0.0,683.89,683.89,1,Direct
1,BK100001,2025,0.0,0.0,27.81,0.0,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.0,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.0,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.0,0.0,743.32,743.32,1,Direct
5,BK100005,2026,0.0,0.0,0.00,0.0,0.0,1395.40,1395.40,1,Direct
6,BK100006,2025,0.0,0.0,0.00,0.0,0.0,1339.30,1339.30,2,Connecting
7,BK100007,2025,0.0,0.0,0.00,0.0,0.0,865.09,865.09,1,Direct
8,BK100008,2026,0.0,0.0,0.00,0.0,0.0,562.55,562.55,1,Direct


In [90]:
bookings_routes[bookings_routes["Booking_ID"].duplicated(keep = False)]

,Booking_ID,Route_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total
6,BK100006,YWG-YYC,2025,0.0,0.00,0.00,0.0,0.0,824.53,824.53
7,BK100006,YYC-YVR,2025,0.0,0.00,0.00,0.0,0.0,514.77,514.77
11,BK100010,YVR-YYC,2026,0.0,0.00,0.00,0.0,0.0,834.79,834.79
12,BK100010,YYC-YWG,2026,0.0,0.00,0.00,0.0,0.0,1107.79,1107.79
39,BK100037,YVR-YYZ,2025,0.0,0.00,7.23,0.0,0.0,1811.96,1819.19
...,...,...,...,...,...,...,...,...,...,...
369753,BK452950,YYZ-YHZ,2025,0.0,0.00,0.00,0.0,0.0,752.51,752.51
369780,BK452977,YYC-YQR,2025,0.0,0.00,0.00,0.0,0.0,675.88,675.88
369781,BK452977,YYC-YQR,2026,0.0,39.48,0.00,0.0,0.0,0.00,39.48
369796,BK452992,YVR-YYZ,2025,0.0,0.00,0.00,0.0,0.0,1545.27,1545.27


## Gate Lease Preparation

In [91]:
# --- 2024-2025 LEASE YEAR ---

# Filter gates leased during January-December 2024
gates_2425 = gates[gates["Year"] == "2024-2025"].copy()

# Remove duplicate gates that are shared across multiple routes
gates_2425 = gates_2425.drop_duplicates(
    subset=["City", "Gate_Number"]
)

# Calculate annual gate costs for each airport
airport_costs_2425 = (
    gates_2425
    .groupby("City")["Annual_Gate_Cost"]
    .sum()
    .reset_index()
)


# --- 2025-2026 LEASE YEAR ---

# Filter gates leased during January-December 2025
gates_2526 = gates[gates["Year"] == "2025-2026"].copy()

# Remove duplicate gates shared across multiple routes
gates_2526 = gates_2526.drop_duplicates(
    subset=["City", "Gate_Number"]
)

# Calculate annual gate costs for each airport
airport_costs_2526 = (
    gates_2526
    .groupby("City")["Annual_Gate_Cost"]
    .sum()
    .reset_index()
)


# --- PREPARE DATA FOR FISCAL YEAR CALCULATIONS ---

# Merge the lease-year costs so each airport has costs for both years
airport_costs = airport_costs_2425.merge(
    airport_costs_2526,
    on="City",
    suffixes=("_2425", "_2526"),
    validate="one_to_one"
)

# Calculate the percentage change in gate costs at each airport
airport_costs["Growth_Rate"] = (
    airport_costs["Annual_Gate_Cost_2526"]
    / airport_costs["Annual_Gate_Cost_2425"] - 1
)

# Estimate missing 2026-2027 lease costs
# Assume each airport's annual percentage change continues
airport_costs["Estimated_2627"] = (
    airport_costs["Annual_Gate_Cost_2526"]
    * (1 + airport_costs["Growth_Rate"])
)

# Display the prepared data
display(airport_costs.head())

,City,Annual_Gate_Cost_2425,Annual_Gate_Cost_2526,Growth_Rate,Estimated_2627
0,Abbotsford,166800,170200,0.020384,1.736693e+05
1,Calgary,1065800,1090600,0.023269,1.115977e+06
2,Edmonton,490700,500800,0.020583,5.111079e+05
3,Halifax,700200,714800,0.020851,7.297044e+05
4,Kelowna,203800,207200,0.016683,2.106567e+05


## Fiscal Year 2024-2025

### Firm Costs

#### Gates

In [92]:
# FY2024-25: Sep-Dec 2024 + Jan-Aug 2025
gate_total_fy_2425 = (
    airport_costs["Annual_Gate_Cost_2425"].sum() * (4/12)
    + airport_costs["Annual_Gate_Cost_2526"].sum() * (8/12)
)

print(f"FY2024-25 gate costs: ${gate_total_fy_2425:,.2f}")

FY2024-25 gate costs: $9,806,600.00


#### Flights

In [93]:
flight_costs.dtypes

Cost_Date       object
Route_ID        object
From            object
To              object
Crew           float64
Food           float64
Fuel           float64
Maintenance    float64
Total          float64
dtype: object

In [94]:
# Select flights from September 2024 through August 2025
flights_2425 = flight_costs[
    (flight_costs["Cost_Date"] >= "2024-09-01") &
    (flight_costs["Cost_Date"] < "2025-09-01")
]
flights_2425_total_costs = flights_2425["Total"].sum().round(2)
flights_2425_total_crew = flights_2425["Crew"].sum().round(2)
flights_2425_total_fuel = flights_2425["Fuel"].sum().round(2)
flights_2425_total_food = flights_2425["Food"].sum().round(2)
flights_2425_total_maintenance = flights_2425["Maintenance"].sum().round(2)

print(f"Total costs for 2024-2025 were: ${flights_2425_total_costs}")
print(f"Total crew costs for 2024-2025 were: ${flights_2425_total_crew}")
print(f"Total fuel costs for 2024-2025 were: ${flights_2425_total_fuel}")
print(f"Total food costs for 2024-2025 were: ${flights_2425_total_food}")
print(f"Total maintenance costs for 2024-2025 were: ${flights_2425_total_maintenance}")


Total costs for 2024-2025 were: $192985563.99
Total crew costs for 2024-2025 were: $59717729.49
Total fuel costs for 2024-2025 were: $67061083.08
Total food costs for 2024-2025 were: $7822627.71
Total maintenance costs for 2024-2025 were: $58384123.71


#### All Costs

In [95]:
total_costs_2425 = gate_total_fy_2425 + flights_2425_total_costs
print(f"Total costs for the fiscal year 2024-2025: ${total_costs_2425}")

Total costs for the fiscal year 2024-2025: $202792163.99


### Firm Revenue

In [96]:
bookings.head(10)

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.00,0.0,0.0,683.89,683.89,1,Direct
1,BK100001,2025,0.0,0.0,27.81,0.0,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.0,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.0,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.0,0.0,743.32,743.32,1,Direct
5,BK100005,2026,0.0,0.0,0.00,0.0,0.0,1395.40,1395.40,1,Direct
6,BK100006,2025,0.0,0.0,0.00,0.0,0.0,1339.30,1339.30,2,Connecting
7,BK100007,2025,0.0,0.0,0.00,0.0,0.0,865.09,865.09,1,Direct
8,BK100008,2026,0.0,0.0,0.00,0.0,0.0,562.55,562.55,1,Direct
9,BK100009,2025,0.0,0.0,0.00,0.0,0.0,934.58,934.58,1,Direct


In [97]:
bookings_2425 = bookings[bookings["Fiscal_Year"] == 2025]
bookings_2425.head(10)

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
1,BK100001,2025,0.0,0.0,27.81,0.00,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.00,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.00,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.00,0.0,743.32,743.32,1,Direct
6,BK100006,2025,0.0,0.0,0.00,0.00,0.0,1339.30,1339.30,2,Connecting
7,BK100007,2025,0.0,0.0,0.00,0.00,0.0,865.09,865.09,1,Direct
9,BK100009,2025,0.0,0.0,0.00,0.00,0.0,934.58,934.58,1,Direct
11,BK100011,2025,0.0,0.0,28.08,0.00,0.0,1364.45,1392.53,1,Direct
13,BK100013,2025,0.0,0.0,0.00,0.00,0.0,1150.43,1150.43,1,Direct
14,BK100014,2025,0.0,0.0,0.00,-1159.95,0.0,1159.95,0.00,1,Direct


In [98]:
# 2024-2025 revenue streams

# total 
bookings_total_cost_2425 = bookings_2425["Total"].sum().round(2)
# change fee
change_fee_2425 = bookings_2425["Change Fee"].sum().round(2)
# extra baggage
baggage_2425 = bookings_2425["Extra Baggage"].sum().round(2)
# food and onboard
foodboard_2425 = bookings_2425["Food & Onboard"].sum().round(2)
# refund
refund_2425 = bookings_2425["Refund"].sum().round(2)
# seat upgrades
upgrades_2425 = bookings_2425["Seat Upgrade"].sum().round(2)
# ticket
ticket_2425 = bookings_2425["Ticket"].sum().round(2)

In [99]:
# total
print(f"Total revenue for 2025 fiscal year: ${bookings_total_cost_2425}")
# change
print(f"Total revenue from changing flights for 2025 fiscal year: ${change_fee_2425}")
# extra baggage
print(f"Total revenue from extra baggage for 2025 fiscal year: ${baggage_2425}")
# food
print(f"Total revenue from food for 2025 fiscal year: ${foodboard_2425}")
# refund
print(f"Total refunds for 2025 fiscal year: ${refund_2425}")
# upgrade
print(f"Total revenue from upgrades for 2025 fiscal year: ${upgrades_2425}")
# ticket
print(f"Total revenue from tickets for 2025 fiscal year: ${ticket_2425}")


Total revenue for 2025 fiscal year: $180629746.3
Total revenue from changing flights for 2025 fiscal year: $558125.0
Total revenue from extra baggage for 2025 fiscal year: $2001639.26
Total revenue from food for 2025 fiscal year: $435546.3
Total refunds for 2025 fiscal year: $-6684921.18
Total revenue from upgrades for 2025 fiscal year: $678561.23
Total revenue from tickets for 2025 fiscal year: $183640795.69


##### Connecting vs Direct

In [100]:
bookings_c2425 = bookings_2425[bookings_2425["Booking_Type"] == "Connecting"]
bookings_d2425 = bookings_2425[bookings_2425["Booking_Type"] == "Direct"]

In [101]:
bookings_c2425["Total"].sum()

np.float64(12449837.01)

In [102]:
bookings_d2425["Total"].sum()

np.float64(168179909.29)

In [103]:
print(f"There are {bookings_d2425.shape[0]} direct flights and {bookings_c2425.shape[0]} connecting")
prop_c2425 = (bookings_c2425.shape[0] / bookings_2425.shape[0]) * 100
prop_d2425 = (bookings_d2425.shape[0] / bookings_2425.shape[0]) * 100
print(f"Proportion of direct flights: {round(prop_d2425, 2)}%")
print(f"Proportion of connecting flights: {round(prop_c2425, 2)}%")

There are 165615 direct flights and 6348 connecting
Proportion of direct flights: 96.31%
Proportion of connecting flights: 3.69%


In [104]:
revenue_prop_c2425 = (bookings_c2425["Total"].sum() / bookings_2425["Total"].sum()) * 100
revenue_prop_d2425 = (bookings_d2425["Total"].sum() / bookings_2425["Total"].sum()) * 100

print(f"Proportion of revenue from direct flights: {round(revenue_prop_d2425, 2)}%")
print(f"Proportion of revenue connecting flights: {round(revenue_prop_c2425, 2)}%")

Proportion of revenue from direct flights: 93.11%
Proportion of revenue connecting flights: 6.89%


## Fiscal Year 2025-2026

### Firm Costs

#### Gates

In [105]:
# FY2025-26: Sep-Dec 2025 + Jan-Aug 2026
gate_total_fy_2526 = (
    airport_costs["Annual_Gate_Cost_2526"].sum() * (4/12)
    + airport_costs["Estimated_2627"].sum() * (8/12)
)

print(f"FY2025-26 gate costs: ${gate_total_fy_2526:,.2f}")

FY2025-26 gate costs: $10,023,125.91


#### Flights

In [106]:
flights_2526 = flights_2526 = flight_costs[
    (flight_costs["Cost_Date"] >= "2025-09-01") &
    (flight_costs["Cost_Date"] < "2026-09-01")
]
flights_2526_total_costs = flights_2526["Total"].sum().round(2)
flights_2526_total_crew = flights_2526["Crew"].sum().round(2)
flights_2526_total_fuel = flights_2526["Fuel"].sum().round(2)
flights_2526_total_food = flights_2526["Food"].sum().round(2)
flights_2526_total_maintenance = flights_2526["Maintenance"].sum().round(2)

print(f"Total costs for 2025-2026 were: ${flights_2526_total_costs}")
print(f"Total crew costs for 2025-2026 were: ${flights_2526_total_crew}")
print(f"Total fuel costs for 2025-2026 were: ${flights_2526_total_fuel}")
print(f"Total food costs for 2025-2026 were: ${flights_2526_total_food}")
print(f"Total maintenance costs for 2025-2026 were: ${flights_2526_total_maintenance}")

Total costs for 2025-2026 were: $238376476.25
Total crew costs for 2025-2026 were: $74767150.35
Total fuel costs for 2025-2026 were: $83182330.22
Total food costs for 2025-2026 were: $9039515.73
Total maintenance costs for 2025-2026 were: $71387479.95


#### All Costs

In [107]:
total_costs_2526 = gate_total_fy_2526 + flights_2526_total_costs
print(f"Total costs for the fiscal year 2025-2026: ${total_costs_2526}")

Total costs for the fiscal year 2025-2026: $248399602.1578863


### Firm Revenue

In [108]:
bookings.head()

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.00,0.0,0.0,683.89,683.89,1,Direct
1,BK100001,2025,0.0,0.0,27.81,0.0,0.0,558.37,586.18,1,Direct
2,BK100002,2025,0.0,0.0,0.00,0.0,0.0,796.89,796.89,1,Direct
3,BK100003,2025,0.0,0.0,0.00,0.0,0.0,596.12,596.12,1,Direct
4,BK100004,2025,0.0,0.0,0.00,0.0,0.0,743.32,743.32,1,Direct


In [109]:
bookings_2526 = bookings[bookings["Fiscal_Year"] == 2026].copy()
bookings_2526.head()

,Booking_ID,Fiscal_Year,Change Fee,Extra Baggage,Food & Onboard,Refund,Seat Upgrade,Ticket,Total,Num_Routes,Booking_Type
0,BK100000,2026,0.0,0.0,0.0,0.0,0.00,683.89,683.89,1,Direct
5,BK100005,2026,0.0,0.0,0.0,0.0,0.00,1395.40,1395.40,1,Direct
8,BK100008,2026,0.0,0.0,0.0,0.0,0.00,562.55,562.55,1,Direct
10,BK100010,2026,0.0,0.0,0.0,0.0,0.00,1942.58,1942.58,2,Connecting
12,BK100012,2026,0.0,0.0,12.1,0.0,58.08,1051.87,1122.05,1,Direct


In [110]:
# 2024-2025 revenue streams

# total 
bookings_total_cost_2526 = bookings_2526["Total"].sum().round(2)
# change fee
change_fee_2526 = bookings_2526["Change Fee"].sum().round(2)
# extra baggage
baggage_2526 = bookings_2526["Extra Baggage"].sum().round(2)
# food and onboard
foodboard_2526 = bookings_2526["Food & Onboard"].sum().round(2)
# refund
refund_2526 = bookings_2526["Refund"].sum().round(2)
# seat upgrades
upgrades_2526 = bookings_2526["Seat Upgrade"].sum().round(2)
# ticket
ticket_2526 = bookings_2526["Ticket"].sum().round(2)

In [111]:
# total
print(f"Total revenue for 2026 fiscal year: ${bookings_total_cost_2526}")
# change
print(f"Total revenue from changing flights for 2026 fiscal year: ${change_fee_2526}")
# extra baggage
print(f"Total revenue from extra baggage for 2026 fiscal year: ${baggage_2526}")
# food
print(f"Total revenue from food for 2026 fiscal year: ${foodboard_2526}")
# refund
print(f"Total refunds for 2026 fiscal year: ${refund_2526}")
# upgrade
print(f"Total revenue from upgrades for 2026 fiscal year: ${upgrades_2526}")
# ticket
print(f"Total revenue from tickets for 2026 fiscal year: ${ticket_2526}")

Total revenue for 2026 fiscal year: $189999278.55
Total revenue from changing flights for 2026 fiscal year: $603975.0
Total revenue from extra baggage for 2026 fiscal year: $2139239.17
Total revenue from food for 2026 fiscal year: $493660.5
Total refunds for 2026 fiscal year: $-7834052.51
Total revenue from upgrades for 2026 fiscal year: $729536.24
Total revenue from tickets for 2026 fiscal year: $193866920.15


#### Connecting vs Direct

In [112]:
bookings_c2526 = bookings_2526[bookings_2526["Booking_Type"] == "Connecting"].copy()
bookings_d2526 = bookings_2526[bookings_2526["Booking_Type"] == "Direct"].copy()

In [113]:
bookings_c2526["Total"].sum()

np.float64(13226441.280000001)

In [114]:
bookings_d2526["Total"].sum()


np.float64(176772837.26999998)

In [115]:
print(f"There are {bookings_d2526.shape[0]} direct flights and {bookings_c2526.shape[0]} connecting")
prop_c2526 = (bookings_c2526.shape[0] / bookings_2526.shape[0]) * 100
prop_d2526 = (bookings_d2526.shape[0] / bookings_2526.shape[0]) * 100
print(f"Proportion of direct flights: {round(prop_d2526, 2)}%")
print(f"Proportion of connecting flights: {round(prop_c2526, 2)}%")

There are 178147 direct flights and 6757 connecting
Proportion of direct flights: 96.35%
Proportion of connecting flights: 3.65%


In [116]:
revenue_prop_c2526 = (bookings_c2526["Total"].sum() / bookings_2526["Total"].sum()) * 100
revenue_prop_d2526 = (bookings_d2526["Total"].sum() / bookings_2526["Total"].sum()) * 100

print(f"Proportion of revenue from direct flights: {round(revenue_prop_d2526, 2)}%")
print(f"Proportion of revenue connecting flights: {round(revenue_prop_c2526, 2)}%")

Proportion of revenue from direct flights: 93.04%
Proportion of revenue connecting flights: 6.96%


## Changes from 2024-2025 to 2025-2026

### Firm Costs

#### Gates

In [117]:
print(f"NorthByte increased spending on gates by: ${gate_total_fy_2526 - gate_total_fy_2425}")

print(f"Increasing their spending by on gates by: {(((gate_total_fy_2526 - gate_total_fy_2425)/gate_total_fy_2425) * 100).round(2)}%")

NorthByte increased spending on gates by: $216525.90788628906
Increasing their spending by on gates by: 2.21%


#### Flights

In [118]:
# flights
print(f"NorthByte increased spending on flights by: ${(flights_2526_total_costs - flights_2425_total_costs).round(2)}")
print(f"Increasing their spending by on flights by: {(((flights_2526_total_costs - flights_2425_total_costs)/flights_2425_total_costs) * 100).round(2)}%")

# crew
print(f"NorthByte increased spending on crew by: ${(flights_2526_total_crew - flights_2425_total_crew).round(2)}")
print(f"Increasing their spending by on crew by: {(((flights_2526_total_crew - flights_2425_total_crew)/flights_2425_total_crew) * 100).round(2)}%")
# fuel
print(f"NorthByte increased spending on fuel by: ${(flights_2526_total_fuel - flights_2425_total_fuel).round(2)}")
print(f"Increasing their spending by on fuel by: {(((flights_2526_total_fuel - flights_2425_total_fuel)/flights_2425_total_fuel) * 100).round(2)}%")
# food
print(f"NorthByte increased spending on food by: ${(flights_2526_total_food - flights_2425_total_food).round(2)}")
print(f"Increasing their spending by on food by: {(((flights_2526_total_food - flights_2425_total_food)/flights_2425_total_food) * 100).round(2)}%")
# maintenance
print(f"NorthByte increased spending on maintenance by: ${(flights_2526_total_maintenance - flights_2425_total_maintenance).round(2)}")
print(f"Increasing their spending by on maintenance by: {(((flights_2526_total_maintenance - flights_2425_total_maintenance)/flights_2425_total_maintenance) * 100).round(2)}%")


NorthByte increased spending on flights by: $45390912.26
Increasing their spending by on flights by: 23.52%
NorthByte increased spending on crew by: $15049420.86
Increasing their spending by on crew by: 25.2%
NorthByte increased spending on fuel by: $16121247.14
Increasing their spending by on fuel by: 24.04%
NorthByte increased spending on food by: $1216888.02
Increasing their spending by on food by: 15.56%
NorthByte increased spending on maintenance by: $13003356.24
Increasing their spending by on maintenance by: 22.27%


#### All Costs

In [119]:
print(f"Costs for the fiscal year 2024-2025 to 2025-2026 increased by a total of: ${(total_costs_2526 - total_costs_2425).round(2)}")
print(f"Increasing by: {(((total_costs_2526 - total_costs_2425)/total_costs_2425)*100).round(2)}%")


Costs for the fiscal year 2024-2025 to 2025-2026 increased by a total of: $45607438.17
Increasing by: 22.49%


#### Firm Revenue

### Visualizations

In [120]:
revenue_by_year = bookings.groupby("Fiscal_Year", as_index=False)["Total"].sum()

alt.Chart(revenue_by_year).mark_bar().encode(
    x="Fiscal_Year:O",
    y="Total:Q",
    color="Fiscal_Year:N"
)

alt.Chart(...)

In [121]:
financial_summary = pd.DataFrame({
    "Fiscal_Year": [2025, 2026],
    "Revenue": [
        bookings_total_cost_2425,
        bookings_total_cost_2526
    ],
    "Costs": [
        total_costs_2425,
        total_costs_2526
    ]
})

financial_long = financial_summary.melt(
    id_vars="Fiscal_Year",
    var_name="Category",
    value_name="Amount"
)

In [122]:
alt.Chart(financial_long).mark_bar().encode(
    x="Fiscal_Year:O",
    y="Amount:Q",
    color="Category:N",
    xOffset="Category:N",
    tooltip=["Fiscal_Year", "Category", "Amount"]
).properties(
    title="Revenue vs Costs by Fiscal Year"
)

alt.Chart(...)

In [123]:
financial_summary["Profit"] = (
    financial_summary["Revenue"] - financial_summary["Costs"]
)

financial_summary["Profit_Margin"] = (
    financial_summary["Profit"] / financial_summary["Revenue"] * 100
)

financial_summary

,Fiscal_Year,Revenue,Costs,Profit,Profit_Margin
0,2025,1.806297e+08,2.027922e+08,-2.216242e+07,-12.269528
1,2026,1.899993e+08,2.483996e+08,-5.840032e+07,-30.737129


In [124]:
revenue_categories = pd.DataFrame({
    "Fiscal_Year": [2025, 2026],
    "Ticket": [ticket_2425, ticket_2526],
    "Extra Baggage": [baggage_2425, baggage_2526],
    "Seat Upgrade": [upgrades_2425, upgrades_2526],
    "Change Fee": [change_fee_2425, change_fee_2526],
    "Food & Onboard": [foodboard_2425, foodboard_2526],
    "Refund": [refund_2425, refund_2526]
})

cost_categories = pd.DataFrame({
    "Fiscal_Year": [2025, 2026],
    "Fuel": [flights_2425_total_fuel, flights_2526_total_fuel],
    "Crew": [flights_2425_total_crew, flights_2526_total_crew],
    "Maintenance": [flights_2425_total_maintenance, flights_2526_total_maintenance],
    "Food": [flights_2425_total_food, flights_2526_total_food],
    "Gates": [gate_total_fy_2425, gate_total_fy_2526]
})

In [125]:
revenue_long = revenue_categories.melt(
    id_vars="Fiscal_Year",
    var_name="Category",
    value_name="Amount"
)

alt.Chart(revenue_long).mark_bar().encode(
    x="Category:N",
    y="Amount:Q",
    color="Fiscal_Year:N",
    xOffset="Fiscal_Year:N",
    tooltip=["Category", "Fiscal_Year", "Amount"]
).properties(
    title="Revenue by Category",
    width=600
)

alt.Chart(...)

In [126]:
revenue_change = revenue_categories.set_index("Fiscal_Year").T

# Calculate difference between years
revenue_change["Change"] = revenue_change[2026] - revenue_change[2025]

# Convert index into a Category column
revenue_change = revenue_change.reset_index().rename(
    columns={"index": "Category"}
)

# Plot only the necessary columns
alt.Chart(revenue_change[["Category", "Change"]]).mark_bar().encode(
    y=alt.Y("Category:N", sort="-x"),
    x=alt.X("Change:Q", axis=alt.Axis(format="~s")),
    color=alt.condition(
        alt.datum.Change >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=[
        "Category",
        alt.Tooltip("Change:Q", format="$,.0f")
    ]
).properties(
    title="Change in Revenue by Category",
    width=550,
    height=300
)

alt.Chart(...)

In [127]:
ancillary_change = revenue_change[
    revenue_change["Category"] != "Ticket"
]

alt.Chart(ancillary_change[["Category", "Change"]]).mark_bar().encode(
    y=alt.Y("Category:N", sort="-x", title=None),
    x=alt.X(
        "Change:Q",
        title="Change in Revenue ($)",
        axis=alt.Axis(format="~s")
    ),
    color=alt.condition(
        alt.datum.Change >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=[
        "Category",
        alt.Tooltip("Change:Q", format="$,.0f")
    ]
).properties(
    title="Change in Ancillary Revenue and Refunds",
    width=550,
    height=250
)

alt.Chart(...)

In [128]:
revenue_change["Percent_Change"] = (
    revenue_change["Change"] / revenue_change[2025].abs() * 100
)


In [129]:
cost_long = cost_categories.melt(
    id_vars="Fiscal_Year",
    var_name="Category",
    value_name="Amount"
)

alt.Chart(cost_long).mark_bar().encode(
    x="Category:N",
    y="Amount:Q",
    color="Fiscal_Year:N",
    xOffset="Fiscal_Year:N",
    tooltip=["Category", "Fiscal_Year", "Amount"]
).properties(
    title="Operating Costs by Category",
    width=600
)

alt.Chart(...)

## More analysis

In [130]:
flight_costs["Cost_Date"] = pd.to_datetime(flight_costs["Cost_Date"])

flight_costs["Fiscal_Year"] = (
    flight_costs["Cost_Date"].dt.year
    + (flight_costs["Cost_Date"].dt.month >= 9).astype(int)
)

flight_summary = (
    flight_costs.groupby("Fiscal_Year")
    .agg(
        Num_Flights=("Route_ID", "size"),
        Total_Cost=("Total", "sum"),
        Avg_Cost_Per_Flight=("Total", "mean")
    )
    .reset_index()
)

flight_summary

,Fiscal_Year,Num_Flights,Total_Cost,Avg_Cost_Per_Flight
0,2025,13072,1.929856e+08,14763.277539
1,2026,15729,2.383765e+08,15155.221327


In [131]:
alt.Chart(flight_summary).mark_bar().encode(
    x="Fiscal_Year:O",
    y="Avg_Cost_Per_Flight:Q",
    color="Fiscal_Year:N",
    tooltip=["Fiscal_Year", "Num_Flights", "Avg_Cost_Per_Flight"]
).properties(
    title="Average Operating Cost per Flight",
    width=350
)

alt.Chart(...)

In [132]:
route_costs = (
    flight_costs.groupby("Route_ID")
    .agg(
        Total_Cost=("Total", "sum"),
        Avg_Cost=("Total", "mean"),
        Num_Flights=("Route_ID", "size")
    )
    .reset_index()
)

route_costs.sort_values("Total_Cost", ascending=False).head(10)

,Route_ID,Total_Cost,Avg_Cost,Num_Flights
48,YYZ-YVR,33885592.20,56855.020470,596
17,YUL-YVR,32965897.76,62553.885693,527
26,YVR-YYZ,30967700.03,50518.270848,613
21,YVR-YUL,30028936.27,55609.141241,540
51,YYZ-YYC,16341656.46,46557.425812,351
6,YHZ-YYZ,16331171.69,25637.632166,637
18,YUL-YYZ,15045254.15,13627.947600,1104
41,YYC-YYZ,14484912.85,41985.254638,345
47,YYZ-YUL,14133474.95,12675.762287,1115
45,YYZ-YHZ,13566427.13,23032.983243,589


In [133]:
route_revenue = (
    bookings_routes.groupby(["Fiscal_Year", "Route_ID"])
    .agg(Revenue=("Total", "sum"))
    .reset_index()
)

route_costs_yearly = (
    flight_costs.groupby(["Fiscal_Year", "Route_ID"])
    .agg(Operating_Cost=("Total", "sum"))
    .reset_index()
)
route_revenue = route_revenue[
    route_revenue["Fiscal_Year"].isin([2025, 2026])
]
route_profit = route_revenue.merge(
    route_costs_yearly,
    on=["Fiscal_Year", "Route_ID"],
    how="outer",
    validate="one_to_one"
)

route_profit[["Revenue", "Operating_Cost"]] = (
    route_profit[["Revenue", "Operating_Cost"]].fillna(0)
)

route_profit["Operating_Profit"] = (
    route_profit["Revenue"] - route_profit["Operating_Cost"]
)

In [134]:
route_profit[route_profit["Fiscal_Year"] == 2026].sort_values(
    "Operating_Profit"
).head(10)

,Fiscal_Year,Route_ID,Revenue,Operating_Cost,Operating_Profit
71,2026,YUL-YVR,9317689.66,19629342.96,-10311653.30
75,2026,YVR-YUL,8495080.07,17889173.63,-9394093.56
60,2026,YHZ-YYZ,4768353.23,9764294.01,-4995940.78
72,2026,YUL-YYZ,4427618.74,8791986.88,-4364368.14
99,2026,YYZ-YHZ,3977148.99,8129783.93,-4152634.94
101,2026,YYZ-YUL,4189650.03,8249976.29,-4060326.26
64,2026,YOW-YYZ,3456661.62,7157163.61,-3700501.99
100,2026,YYZ-YOW,3305145.75,6798697.29,-3493551.54
58,2026,YHZ-YUL,2009227.95,4203219.47,-2193991.52
68,2026,YUL-YHZ,1857125.18,3886459.48,-2029334.30


In [135]:
worst_routes = (
    route_profit[route_profit["Fiscal_Year"] == 2026]
    .nsmallest(10, "Operating_Profit")
)

alt.Chart(worst_routes).mark_bar().encode(
    y=alt.Y("Route_ID:N", sort="-x"),
    x=alt.X("Operating_Profit:Q", axis=alt.Axis(format="~s")),
    color=alt.condition(
        alt.datum.Operating_Profit >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=["Route_ID", "Operating_Profit"]
).properties(
    title="10 Least Profitable Routes — FY2026",
    width=500
)

alt.Chart(...)

In [136]:
airport_costs["FY2025_Gate_Cost"] = (
    airport_costs["Annual_Gate_Cost_2425"] * (4/12)
    + airport_costs["Annual_Gate_Cost_2526"] * (8/12)
)

airport_costs["FY2026_Gate_Cost"] = (
    airport_costs["Annual_Gate_Cost_2526"] * (4/12)
    + airport_costs["Estimated_2627"] * (8/12)
)

In [137]:
airport_costs[
    ["City", "FY2025_Gate_Cost", "FY2026_Gate_Cost"]
]

,City,FY2025_Gate_Cost,FY2026_Gate_Cost
0,Abbotsford,1.690667e+05,1.725129e+05
1,Calgary,1.082333e+06,1.107518e+06
2,Edmonton,4.974333e+05,5.076719e+05
3,Halifax,7.099333e+05,7.247363e+05
4,Kelowna,2.060667e+05,2.095045e+05
5,London,1.649667e+05,1.671864e+05
6,Moncton,1.571333e+05,1.594560e+05
7,Montreal,1.142867e+06,1.171332e+06
8,Ottawa,4.365000e+05,4.443940e+05
9,Prince George,1.582333e+05,1.605558e+05


In [138]:
print(airport_costs["FY2025_Gate_Cost"].sum())
print(airport_costs["FY2026_Gate_Cost"].sum())

9806600.0
10023125.907886289


In [139]:
airport_flights = (
    flight_costs.groupby(["Fiscal_Year", "From"])
    .size()
    .reset_index(name="Num_Flights")
)

airport_flights.head()

,Fiscal_Year,From,Num_Flights
0,2025,Abbotsford,152
1,2025,Calgary,1644
2,2025,Edmonton,686
3,2025,Halifax,647
4,2025,Kelowna,554


In [140]:
gate_costs_long = airport_costs.melt(
    id_vars="City",
    value_vars=["FY2025_Gate_Cost", "FY2026_Gate_Cost"],
    var_name="Fiscal_Year",
    value_name="Gate_Cost"
)

gate_costs_long["Fiscal_Year"] = (
    gate_costs_long["Fiscal_Year"]
    .str.extract(r"FY(\d{4})")
    .astype(int)
)

airport_gate_allocation = gate_costs_long.merge(
    airport_flights,
    left_on=["City", "Fiscal_Year"],
    right_on=["From", "Fiscal_Year"],
    how="left",
    validate="one_to_one"
)
airport_gate_allocation["Gate_Cost_Per_Flight"] = (
    airport_gate_allocation["Gate_Cost"]
    / airport_gate_allocation["Num_Flights"]
)



In [141]:
airport_gate_allocation[
    ["City", "Fiscal_Year", "Num_Flights", "Gate_Cost_Per_Flight"]
].head()

,City,Fiscal_Year,Num_Flights,Gate_Cost_Per_Flight
0,Abbotsford,2025,152,1112.280702
1,Calgary,2025,1644,658.353609
2,Edmonton,2025,686,725.121477
3,Halifax,2025,647,1097.269449
4,Kelowna,2025,554,371.961492


In [142]:
airport_gate_allocation[
    ["City", "Fiscal_Year", "Num_Flights", "Gate_Cost_Per_Flight"]
].isna().sum()

City                    0
Fiscal_Year             0
Num_Flights             0
Gate_Cost_Per_Flight    0
dtype: int64

In [143]:
flights_with_gates = flight_costs.merge(
    airport_gate_allocation[
        ["From", "Fiscal_Year", "Gate_Cost_Per_Flight"]
    ],
    on=["From", "Fiscal_Year"],
    how="left",
    validate="many_to_one"
)
flights_with_gates[
    ["Fiscal_Year", "Route_ID", "From", "Total", "Gate_Cost_Per_Flight"]
].head()

,Fiscal_Year,Route_ID,From,Total,Gate_Cost_Per_Flight
0,2025,YVR-YLW,Vancouver,4618.25,821.541950
1,2025,YLW-YYC,Kelowna,5593.93,371.961492
2,2025,YYZ-YYC,Toronto,39550.65,742.626183
3,2025,YLW-YVR,Kelowna,4719.07,371.961492
4,2025,YQR-YYC,Regina,8507.67,1063.616558


In [144]:
flights_with_gates["Total_Cost_With_Gates"] = (
    flights_with_gates["Total"]
    + flights_with_gates["Gate_Cost_Per_Flight"]
)

route_costs_with_gates = (
    flights_with_gates[
        flights_with_gates["Fiscal_Year"].isin([2025, 2026])
    ]
    .groupby(["Fiscal_Year", "Route_ID"], as_index=False)
    .agg(
        Total_Cost=("Total_Cost_With_Gates", "sum"),
        Num_Flights=("Route_ID", "size")
    )
)

route_profit_with_gates = route_revenue.merge(
    route_costs_with_gates,
    on=["Fiscal_Year", "Route_ID"],
    how="outer",
    validate="one_to_one"
)

route_profit_with_gates[["Revenue", "Total_Cost"]] = (
    route_profit_with_gates[["Revenue", "Total_Cost"]].fillna(0)
)

route_profit_with_gates["Profit"] = (
    route_profit_with_gates["Revenue"]
    - route_profit_with_gates["Total_Cost"]
)

route_profit_with_gates["Profit_Margin"] = (
    route_profit_with_gates["Profit"]
    / route_profit_with_gates["Revenue"] * 100
)

route_profit_with_gates["Profit_Per_Flight"] = (
    route_profit_with_gates["Profit"]
    / route_profit_with_gates["Num_Flights"]
)

route_profit_with_gates[
    route_profit_with_gates["Fiscal_Year"] == 2026
].sort_values("Profit").head(10)



,Fiscal_Year,Route_ID,Revenue,Total_Cost,Num_Flights,Profit,Profit_Margin,Profit_Per_Flight
71,2026,YUL-YVR,9317689.66,1.981400e+07,306,-1.049631e+07,-112.649325,-34301.681503
75,2026,YVR-YUL,8495080.07,1.814378e+07,314,-9.648698e+06,-113.579836,-30728.337524
60,2026,YHZ-YYZ,4768353.23,1.003390e+07,372,-5.265543e+06,-110.426859,-14154.684618
72,2026,YUL-YYZ,4427618.74,9.170965e+06,628,-4.743346e+06,-107.130862,-7553.098900
101,2026,YYZ-YUL,4189650.03,8.625668e+06,634,-4.436018e+06,-105.880399,-6996.874076
99,2026,YYZ-YHZ,3977148.99,8.333629e+06,344,-4.356480e+06,-109.537772,-12664.187180
100,2026,YYZ-YOW,3305145.75,7.428011e+06,1062,-4.122865e+06,-124.740796,-3882.170536
64,2026,YOW-YYZ,3456661.62,7.448745e+06,1038,-3.992083e+06,-115.489549,-3845.937293
58,2026,YHZ-YUL,2009227.95,4.365560e+06,224,-2.356332e+06,-117.275516,-10519.341285
69,2026,YUL-YOW,958387.74,3.281719e+06,546,-2.323331e+06,-242.420793,-4255.185265


In [145]:
flights_with_gates[
    flights_with_gates["Fiscal_Year"].isin([2025, 2026])
]["Gate_Cost_Per_Flight"].isna().sum()

np.int64(0)

In [146]:
flights_with_gates[
    flights_with_gates["Fiscal_Year"].isin([2025, 2026])
].groupby("Fiscal_Year")["Gate_Cost_Per_Flight"].sum()

Fiscal_Year
2025    9.806600e+06
2026    1.002313e+07
Name: Gate_Cost_Per_Flight, dtype: float64

In [147]:
route_comparison = route_profit_with_gates.pivot(
    index="Route_ID",
    columns="Fiscal_Year",
    values="Profit"
)

route_comparison["Profit_Change"] = (
    route_comparison[2026] - route_comparison[2025]
)

route_comparison = route_comparison.reset_index()

route_comparison.sort_values("Profit_Change").head(10)

Fiscal_Year,Route_ID,2025,2026,Profit_Change
17,YUL-YVR,-4.412914e+06,-1.049631e+07,-6.083400e+06
21,YVR-YUL,-4.023585e+06,-9.648698e+06,-5.625113e+06
6,YHZ-YYZ,-2.257719e+06,-5.265543e+06,-3.007823e+06
45,YYZ-YHZ,-1.799759e+06,-4.356480e+06,-2.556722e+06
18,YUL-YYZ,-2.322374e+06,-4.743346e+06,-2.420972e+06
47,YYZ-YUL,-2.196561e+06,-4.436018e+06,-2.239457e+06
10,YOW-YYZ,-2.051822e+06,-3.992083e+06,-1.940261e+06
46,YYZ-YOW,-2.223705e+06,-4.122865e+06,-1.899160e+06
4,YHZ-YUL,-1.100824e+06,-2.356332e+06,-1.255509e+06
14,YUL-YHZ,-9.664851e+05,-2.166925e+06,-1.200440e+06


In [148]:
worst_changes = route_comparison.nsmallest(
    10, "Profit_Change"
)

alt.Chart(worst_changes[["Route_ID", "Profit_Change"]]).mark_bar().encode(
    y=alt.Y("Route_ID:N", sort="x", title="Route"),
    x=alt.X(
        "Profit_Change:Q",
        title="Change in Profit ($)",
        axis=alt.Axis(format="~s")
    ),
    color=alt.condition(
        alt.datum.Profit_Change >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=[
        "Route_ID",
        alt.Tooltip("Profit_Change:Q", format="$,.0f")
    ]
).properties(
    title="Routes with Largest Profit Declines",
    width=550,
    height=350
)

alt.Chart(...)

In [149]:
cost_change = cost_categories.set_index("Fiscal_Year").T

cost_change["Change"] = (
    cost_change[2026] - cost_change[2025]
)

cost_change = cost_change.reset_index().rename(
    columns={"index": "Category"}
)

alt.Chart(cost_change[["Category", "Change"]]).mark_bar(
    color="#E76F51"
).encode(
    y=alt.Y("Category:N", sort="-x", title=None),
    x=alt.X(
        "Change:Q",
        title="Increase in Cost ($)",
        axis=alt.Axis(format="~s")
    ),
    tooltip=[
        "Category",
        alt.Tooltip("Change:Q", format="$,.0f")
    ]
).properties(
    title="Operating Cost Increases by Category",
    width=550,
    height=300
)

alt.Chart(...)

In [150]:
fy2026_routes = route_profit_with_gates[
    (route_profit_with_gates["Fiscal_Year"] == 2026)
    & (route_profit_with_gates["Num_Flights"] > 0)
].copy()

worst_per_flight = fy2026_routes.nsmallest(
    10, "Profit_Per_Flight"
)

alt.Chart(worst_per_flight).mark_bar(
    color="#E76F51"
).encode(
    y=alt.Y("Route_ID:N", sort="x", title="Route"),
    x=alt.X(
        "Profit_Per_Flight:Q",
        title="Profit per Flight ($)"
    ),
    tooltip=[
        "Route_ID",
        "Num_Flights",
        alt.Tooltip("Profit_Per_Flight:Q", format="$,.0f")
    ]
).properties(
    title="Least Profitable Routes per Flight — FY2026",
    width=550,
    height=350
)

alt.Chart(...)

In [151]:
route_scatter = fy2026_routes.copy()

route_scatter["Revenue_Per_Flight"] = (
    route_scatter["Revenue"] / route_scatter["Num_Flights"]
)

route_scatter["Cost_Per_Flight"] = (
    route_scatter["Total_Cost"] / route_scatter["Num_Flights"]
)

max_value = max(
    route_scatter["Revenue_Per_Flight"].max(),
    route_scatter["Cost_Per_Flight"].max()
)

points = alt.Chart(route_scatter).mark_circle(size=90).encode(
    x=alt.X("Revenue_Per_Flight:Q", title="Revenue per Flight ($)"),
    y=alt.Y("Cost_Per_Flight:Q", title="Cost per Flight ($)"),
    color=alt.condition(
        alt.datum.Profit >= 0,
        alt.value("#2A9D8F"),
        alt.value("#E76F51")
    ),
    tooltip=[
        "Route_ID",
        "Num_Flights",
        alt.Tooltip("Profit:Q", format="$,.0f")
    ]
)

break_even = alt.Chart(pd.DataFrame({
    "x": [0, max_value],
    "y": [0, max_value]
})).mark_line(strokeDash=[5, 5], color="gray").encode(
    x="x:Q",
    y="y:Q"
)

(points + break_even).properties(
    title="Route Revenue vs Cost per Flight — FY2026",
    width=500,
    height=400
)

alt.LayerChart(...)

In [152]:
Bookings_per_route = bookings_routes.groupby("Route_ID").size().sort_values(ascending = True)

Flights_per_route = flight_costs.groupby("Route_ID").size().sort_values(ascending = True)

route_demand = pd.DataFrame({
    "Flights": Flights_per_route,
    "Tickets": Bookings_per_route,
})

route_demand["Tickets_Per_Flight"] = (route_demand["Tickets"] / route_demand["Flights"]).round(1)
# route_demand.sort_values("Tickets_Per_Flight")

route_demand["Revenue"] = bookings_routes.groupby("Route_ID")["Total"].sum()
route_demand["Cost"] = flight_costs.groupby("Route_ID")["Total"].sum()
route_demand["Profit"] = route_demand["Revenue"] - route_demand["Cost"]

route_demand_long = route_demand.reset_index().melt(
    id_vars= "Route_ID",
    value_vars=["Revenue", "Cost","Profit"],
    var_name= "Type",
    value_name= "Value"
)
# route_demand_long.head()
# route_demand_long.loc[route_demand_long["Type"]== "Cost", "Value"] *= -1

chart = alt.Chart(route_demand_long).mark_bar().encode(
    y=alt.Y("Route_ID:N", title="Route"),
    yOffset=alt.YOffset("Type:N"),
    x=alt.X("Value:Q",stack = None, title="Value"),
    color = alt.Color("Type:N", scale = alt.Scale(
        domain = ["Revenue", "Cost", "Profit"],
        range = ["steelblue", "orange", "green"]
    )),
    order = alt.Order("Type:N", sort = "descending")
).properties(height=700, title="Profit by route")

chart

alt.Chart(...)

In [153]:
np.random.seed(42)

direct_vs_connect = (
    alt.Chart(bookings.sample(5000))
    .mark_boxplot(extent='min-max', size=50)
    .encode(
        x=alt.X('Booking_Type:N', title='Booking Type'),
        y=alt.Y('Total:Q', title='Booking Total Revenue (CAD $)'),
        color=alt.Color(
            'Booking_Type:N',
            legend=None,
        ),
    )
    .properties(
        title='Direct vs. Connecting Bookings',
        width=400,
        height=400,
    )
)

direct_vs_connect

alt.Chart(...)

In [154]:
temp_costs = flight_costs.copy()

temp_costs['Cost_Date'] = pd.to_datetime(temp_costs['Cost_Date'])

daily_costs = (
    temp_costs.groupby(pd.Grouper(key='Cost_Date', freq='W'))['Total']
    .mean()
    .reset_index()
)

# 3. Create a line chart using Altair to spot temporal fluctuations
chart_price_trend = (
    alt.Chart(daily_costs)
    .mark_line(point=False, interpolate='monotone', color='steelblue', strokeWidth=2.5)
    .encode(
        x=alt.X(
            'Cost_Date:T', title='Date'
        ),  # :T tells Altair it's a temporal/date field
        y=alt.Y('Total:Q', title='Average Cost / Price ($)'),
    )
    .properties(
        title='Flight Cost Fluctuation Over Time', width=700, height=350
    )
)

chart_price_trend

alt.Chart(...)

In [155]:
temp_costs = flight_costs.copy()
temp_costs['Cost_Date'] = pd.to_datetime(temp_costs['Cost_Date'])

# 2. Find the top 5 routes by total cost so the chart isn't overcrowded with lines
top_5_routes = (
    temp_costs.groupby('Route_ID')['Total']
    .sum()
    .nlargest(5)
    .index
)
filtered_costs = temp_costs[temp_costs['Route_ID'].isin(top_5_routes)]

# 3. Group weekly by Date and Route_ID
weekly_route_costs = (
    filtered_costs.groupby([pd.Grouper(key='Cost_Date', freq='W'), 'Route_ID'])[
        'Total'
    ]
    .mean()
    .reset_index()
)

# 4. Create a multi-line chart colored by Route ID
chart_route_trend = (
    alt.Chart(weekly_route_costs)
    .mark_line(interpolate='monotone', strokeWidth=2.5)
    .encode(
        x=alt.X('Cost_Date:T', title='Date (Weekly Average)'),
        y=alt.Y('Total:Q', title='Average Operational Cost ($)'),
        color=alt.Color(
            'Route_ID:N', title='Route ID'
        ),  # Colors each route a different line
        tooltip=['Cost_Date:T', 'Route_ID:N', 'Total:Q'],
    )
    .properties(
        title='Weekly Average Cost Trend for Top 5 Most Expensive Routes',
        width=700,
        height=350,
    )
)

chart_route_trend

alt.Chart(...)

In [156]:
print(gates["Route_ID"].unique())
print(flight_costs["Route_ID"].unique())
print(bookings_routes["Route_ID"].unique())

['YXX-YYC' 'YYC-YVR' 'YYC-YYZ' 'YYC-YEG' 'YYC-YWG' 'YYC-YXE' 'YYC-YQR'
 'YYC-YLW' 'YYC-YXX' 'YEG-YVR' 'YEG-YYC' 'YEG-YZF' 'YHZ-YYZ' 'YHZ-YUL'
 'YHZ-YYT' 'YHZ-YQM' nan 'YLW-YVR' 'YLW-YYC' 'YXU-YYZ' 'YQM-YHZ' 'YUL-YVR'
 'YUL-YYZ' 'YUL-YQB' 'YUL-YHZ' 'YUL-YOW' 'YOW-YYZ' 'YOW-YUL' 'YXS-YVR'
 'YQB-YUL' 'YQR-YYC' 'YXE-YYC' 'YYT-YYZ' 'YYT-YHZ' 'YYZ-YVR' 'YYZ-YYC'
 'YYZ-YUL' 'YYZ-YOW' 'YYZ-YHZ' 'YYZ-YWG' 'YYZ-YYT' 'YYZ-YXU' 'YVR-YYZ'
 'YVR-YUL' 'YVR-YYC' 'YVR-YEG' 'YVR-YLW' 'YVR-YYJ' 'YVR-YXY' 'YVR-YXS'
 'YYJ-YVR' 'YXY-YVR' 'YWG-YYC' 'YWG-YYZ' 'YZF-YEG']
['YVR-YLW' 'YLW-YYC' 'YYZ-YYC' 'YLW-YVR' 'YQR-YYC' 'YVR-YUL' 'YWG-YYZ'
 'YUL-YQB' 'YQB-YUL' 'YUL-YYZ' 'YYC-YEG' 'YOW-YYZ' 'YVR-YYC' 'YUL-YVR'
 'YXX-YYC' 'YYC-YXE' 'YVR-YXS' 'YVR-YYJ' 'YYJ-YVR' 'YYZ-YVR' 'YYZ-YUL'
 'YYZ-YHZ' 'YYZ-YOW' 'YYC-YVR' 'YYC-YXX' 'YUL-YHZ' 'YVR-YYZ' 'YXE-YYC'
 'YYC-YWG' 'YYC-YYZ' 'YHZ-YYZ' 'YYZ-YWG' 'YWG-YYC' 'YVR-YEG' 'YEG-YVR'
 'YEG-YYC' 'YVR-YXY' 'YYC-YLW' 'YYT-YHZ' 'YXY-YVR' 'YHZ-YUL' 'YZF-YEG'
 'YYC-YQR' 'YYT-YYZ' 

In [157]:
# NOTE: NEED TO CONFIRM THESE DISTANCES
route_distances = {
    #(YVR) 
    'YVR-YYZ': 3350,
    'YYZ-YVR': 3350,
    'YVR-YUL': 3680,
    'YUL-YVR': 3680,
    'YVR-YYC': 675,
    'YYC-YVR': 675,
    'YVR-YEG': 700,
    'YEG-YVR': 700,
    'YVR-YLW': 275,
    'YLW-YVR': 275,
    'YVR-YYJ': 95,
    'YYJ-YVR': 95,
    'YVR-YXY': 1350,
    'YXY-YVR': 1350,
    'YVR-YXS': 750,
    'YXS-YVR': 750,
    'YVR-YZF': 1300,
    'YZF-YVR': 1300,
    #(YYC) 
    'YYC-YYZ': 2700,
    'YYZ-YYC': 2700,
    'YYC-YEG': 300,
    'YEG-YYC': 300,
    'YYC-YWG': 1330,
    'YWG-YYC': 1330,
    'YYC-YXE': 500,
    'YXE-YYC': 500,
    'YYC-YQR': 500,
    'YQR-YYC': 500,
    'YYC-YLW': 400,
    'YYC-YLW': 400,
    'YYC-YXX': 650,
    'YXX-YYC': 650,
    #(YYZ) 
    'YYZ-YUL': 500,
    'YUL-YYZ': 500,
    'YYZ-YOW': 350,
    'YOW-YYZ': 350,
    'YYZ-YHZ': 1250,
    'YHZ-YYZ': 1250,
    'YYZ-YWG': 1220,
    'YWG-YYZ': 1220,
    'YYZ-YYT': 2100,
    'YYT-YYZ': 2100,
    'YYZ-YXU': 115,
    'YXU-YYZ': 115,
    #(YHZ) & East Coast / Montreal connections
    'YHZ-YQM': 310,
    'YQM-YHZ': 310,
    'YHZ-YUL': 800,
    'YUL-YHZ': 800,
    'YHZ-YYT': 860,
    'YYT-YHZ': 860,
    'YUL-YQB': 240,
    'YQB-YUL': 240,
    'YUL-YOW': 160,
    'YOW-YUL': 160,
    # Other regional connections
    'YEG-YZF': 1050,
    'YZF-YEG': 1050,
}

temp_costs = flight_costs.copy()

temp_costs['Distance_km'] = temp_costs['Route_ID'].map(route_distances)

temp_costs['Cost_Per_Km'] = temp_costs['Total'] / temp_costs['Distance_km']

print(
    temp_costs[['Route_ID', 'Total', 'Distance_km', 'Cost_Per_Km']].head(10)
)

  Route_ID     Total  Distance_km  Cost_Per_Km
0  YVR-YLW   4618.25        275.0    16.793636
1  YLW-YYC   5593.93          NaN          NaN
2  YYZ-YYC  39550.65       2700.0    14.648389
3  YLW-YVR   4719.07        275.0    17.160255
4  YQR-YYC   8507.67        500.0    17.015340
5  YVR-YUL  42748.69       3680.0    11.616492
6  YWG-YYZ  20017.64       1220.0    16.407902
7  YUL-YQB   4610.39        240.0    19.209958
8  YQB-YUL   4271.29        240.0    17.797042
9  YUL-YYZ  11351.95        500.0    22.703900


In [158]:
temp_costs = flight_costs.copy()


temp_costs['Distance_km'] = temp_costs['Route_ID'].map(route_distances)
temp_costs['Distance_km'] = temp_costs[
    'Distance_km'
].fillna(500)  
temp_costs['Cost_Per_Km'] = temp_costs['Total'] / temp_costs['Distance_km']

avg_cost_per_km = (
    temp_costs.groupby('Route_ID')['Cost_Per_Km'].mean().reset_index()
)

top_inefficient_routes = avg_cost_per_km.sort_values(
    by='Cost_Per_Km', ascending=False
).head(15)

chart_cost_per_km = (
    alt.Chart(top_inefficient_routes)
    .mark_bar()
    .encode(
        y=alt.Y('Route_ID:N', title='Route ID', sort='-x'),
        x=alt.X('Cost_Per_Km:Q', title='Average Cost per Kilometer ($/km)'),
        tooltip=['Route_ID:N', alt.Tooltip('Cost_Per_Km:Q', format='$.2f')],
    )
    .properties(
        title='Most Inefficient Routes: Highest Operational Cost per Kilometer',
        width=600,
        height=400,
    )
)

chart_cost_per_km

alt.Chart(...)

In [180]:
transactions = pd.read_csv("../data/processed/transactions_clean.csv", parse_dates=["Transaction_Date"])
transactions["Month"] = transactions["Transaction_Date"].dt.to_period("M").astype(str)

monthly_number_transactions = (
    transactions.groupby("Month")["Transaction_ID"].nunique()
    .reset_index(name="Num_Transactions")
)

monthly_number_transactions_chart = alt.Chart(monthly_number_transactions).mark_line(point=True).encode(
    x=alt.X("Month:T", title="Month"),
    y=alt.Y("Num_Transactions:Q", title="Number of Transactions"),
    tooltip=["Month", "Num_Transactions"]
).properties(
    title="Monthly Number of Transactions",
    width=700,
    height=400
)

monthly_revenue = (
    transactions.groupby("Month")["Amount"].sum()
    .reset_index(name="Total_Revenue")
)

monthly_revenue_chart = alt.Chart(monthly_revenue).mark_line(point=True).encode(
    x=alt.X("Month:T", title="Month"),
    y=alt.Y("Total_Revenue:Q", title="Total Revenue ($)"),
    tooltip=["Month", "Total_Revenue"]
).properties(
    title="Monthly Revenue",
    width=700,
    height=400
)

# compare the two fiscal years, Sep -> Aug 
monthly_combined = monthly_number_transactions.merge(monthly_revenue, on="Month")
monthly_combined = monthly_combined[
    (monthly_combined["Month"] >= "2024-09") & (monthly_combined["Month"] <= "2026-08")
].copy()

month_dates = pd.to_datetime(monthly_combined["Month"])
monthly_combined["Fiscal_Year"] = np.where(month_dates.dt.month >= 9, month_dates.dt.year + 1, month_dates.dt.year)
monthly_combined["FY_Label"] = monthly_combined["Fiscal_Year"].map({2025: "FY 2024-25", 2026: "FY 2025-26"})
monthly_combined["Month_Name"] = month_dates.dt.strftime("%b")

fiscal_month_order = ["Sep", "Oct", "Nov", "Dec", "Jan", "Feb",
                      "Mar", "Apr", "May", "Jun", "Jul", "Aug"]

base_fy = alt.Chart(monthly_combined).encode(
    x=alt.X("Month_Name:N", sort=fiscal_month_order, title="Month"),
    color=alt.Color("FY_Label:N", title="Fiscal year"),
)

transactions_fy = base_fy.mark_line(point=True).encode(
    y=alt.Y("Num_Transactions:Q", title="Number of Transactions"),
    tooltip=[
        alt.Tooltip("FY_Label:N", title="Fiscal year"),
        alt.Tooltip("Month_Name:N", title="Month"),
        alt.Tooltip("Num_Transactions:Q", title="Transactions", format=","),
    ],
).properties(width=700, height=220, title="Monthly Transactions")

revenue_fy = base_fy.mark_line(point=True).encode(
    y=alt.Y("Total_Revenue:Q", title="Total Revenue ($)", axis=alt.Axis(format="$~s")),
    tooltip=[
        alt.Tooltip("FY_Label:N", title="Fiscal year"),
        alt.Tooltip("Month_Name:N", title="Month"),
        alt.Tooltip("Total_Revenue:Q", title="Revenue", format="$,.0f"),
    ],
).properties(width=700, height=220, title="Monthly Revenue")

monthly_fy_chart = alt.vconcat(transactions_fy, revenue_fy).properties(
    title="Transactions and Revenue by Fiscal Year"
)

monthly_fy_chart


alt.VConcatChart(...)